### TCGA / GDC Portal

https://portal.gdc.cancer.gov/

## GDC

https://docs.gdc.cancer.gov/API/Users_Guide/Downloading_Files/

### 🧠 Big picture (this is powerful)

- You’ve essentially built the backbone of:
- cBioPortal-like explorer
- cohort stratification engine
- patient-specific pipeline (your perturbation_agent 🔥)

### 💡 Flow

> project → project_id  
> primary_sites → pid and disease_type  
> subtypes → subtype_id  
> stages →  stage_id  
> case_id → case_ids or UUIDs  
> samples → sample_id [tumor, normal]  
> aliquots → aliquot_id  
> files → file_id  


In [1]:
import os, sys
import pandas as pd

from pathlib import Path

ROOT = Path().resolve().parent.parent
SRC = os.path.join(ROOT, "src")

if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

print("ROOT:", ROOT)
print("SRC added:", SRC)

from libs.tcga_gdc_lib import *
from libs.Basic import *


ROOT: /home/flavio/uv
SRC added: /home/flavio/uv/src


### Defaults

In [2]:
ROOT = Path().resolve().parent
root_data = os.path.join(ROOT, "data/tcga")

gdc = GDC(root_data=root_data)

os.listdir(root_data)


['cases_for_PS_TCGA-BLCA_Subtype_transitional cell carcinoma_Stage_stage iii.tsv',
 'cases_for_PS_TCGA-BLCA_Subtype_transitional cell carcinoma_Stage_stage ii.tsv',
 'samples_for_PS_TCGA-BLCA_Subtype_transitional cell carcinoma_Stage_stage iv_case_aa14ffb4-a715-4636-8251-73d019dddd8c.tsv',
 'samples_for_PS_TCGA-BLCA_Subtype_transitional cell carcinoma_Stage_stage ii.tsv',
 'subtype_for_PS_TCGA-BLCA.tsv',
 'cases_for_PS_TCGA-BLCA_Subtype_transitional cell carcinoma_Stage_stage iv.tsv',
 'samples_for_PS_TCGA-BLCA_Subtype_transitional cell carcinoma_Stage_stage iv.tsv',
 'stage_for_PS_TCGA-BLCA_Subtype_transitional cell carcinoma.tsv',
 'programs.txt',
 'primary_site_program_TCGA.tsv',
 'samples_for_PS_TCGA-BLCA_Subtype_transitional cell carcinoma_Stage_stage iii_case_aa14e232-c6c6-4075-9d15-e1857373d233.tsv']

### All programs

In [3]:
force=False
verbose=True

prog_list = gdc.list_gdc_progams(force=force, verbose=verbose)

# prog_list

File read at '/home/flavio/uv/perturb_agent/data/tcga/programs.txt'


### Get valid subtypes

facets = "diagnoses.primary_diagnosis"

### For each subtype → get stages

facets = "diagnoses.tumor_stage"

### For each (subtype, stage) → get samples

fields = "case_id,submitter_id,diagnoses.tumor_stage"

### Then → fetch RNA-seq files

endpoint = "/files"
field = "cases.case_id"



In [4]:
force=False
verbose=True

program = 'TCGA'

dfc = gdc.get_primary_sites(program=program, force=force, verbose=verbose)

dfc.head(3)


Table opened ((33, 3)) at '/home/flavio/uv/perturb_agent/data/tcga/primary_site_program_TCGA.tsv'


,project_id,primary_site,disease_type
0,TCGA-ACC,['Adrenal gland'],['Adenomas and Adenocarcinomas']
1,TCGA-BLCA,['Bladder'],"['Epithelial Neoplasms, NOS', 'Squamous Cell N..."
2,TCGA-BRCA,['Breast'],"['Adnexal and Skin Appendage Neoplasms', 'Aden..."


### Subtypes

In [5]:
force=False
verbose=True

i=1
pid = dfc.iloc[i].project_id
primary_site = dfc.iloc[i].primary_site

print(pid, primary_site)

df_subt = gdc.build_subtypes(pid=pid, do_filter=True, force=force, verbose=verbose)

print(len(df_subt))
df_subt

TCGA-BLCA ['Bladder']
Table opened ((22, 6)) at '/home/flavio/uv/perturb_agent/data/tcga/subtype_for_PS_TCGA-BLCA.tsv'
9


,project_id,subtype,n,subtype_raw,is_valid,frac
0,TCGA-BLCA,transitional cell carcinoma,348,transitional cell carcinoma,True,0.701613
1,TCGA-BLCA,papillary transitional cell carcinoma,76,papillary transitional cell carcinoma,True,0.153226
2,TCGA-BLCA,"papillary transitional cell carcinoma, non-inv...",66,"papillary transitional cell carcinoma, non-inv...",True,0.133065
3,TCGA-BLCA,acinar adenocarcinoma,1,acinar adenocarcinoma,True,0.002016
4,TCGA-BLCA,basaloid squamous cell carcinoma,1,basaloid squamous cell carcinoma,True,0.002016
5,TCGA-BLCA,merkel cell carcinoma,1,merkel cell carcinoma,True,0.002016
6,TCGA-BLCA,papillary renal cell carcinoma,1,papillary renal cell carcinoma,True,0.002016
7,TCGA-BLCA,"squamous cell carcinoma, clear cell type",1,"squamous cell carcinoma, clear cell type",True,0.002016
8,TCGA-BLCA,transitional cell carcinoma in situ,1,transitional cell carcinoma in situ,True,0.002016


### Stages

In [ ]:
force=False
verbose=True

i=3
subtype = df_subt.iloc[i].subtype_raw

print(pid, subtype)

df_stage = gdc.build_stages(pid=pid, subtype=subtype, do_filter=True, force=force, verbose=verbose)

print(len(df_stage))
df_stage

TCGA-BLCA acinar adenocarcinoma
Table saved ((2, 7)) at '/home/flavio/uv/perturb_agent/data/tcga/stage_for_PS_TCGA-BLCA_Subtype_acinar adenocarcinoma.tsv'
2


,project_id,subtype,stage,n,stage_raw,is_valid,frac
0,TCGA-BLCA,acinar adenocarcinoma,stage ii,1,stage ii,True,0.5
1,TCGA-BLCA,acinar adenocarcinoma,stage iia,1,stage iia,True,0.5


In [15]:
force=False
verbose=True

i=1
stage = df_stage.iloc[i].stage_raw


print(pid, subtype, stage)

df_case = gdc.build_cases(pid=pid, subtype=subtype, stage=stage, force=force, verbose=verbose)
print(len(df_case))
df_case.head(3)

TCGA-BLCA acinar adenocarcinoma stage iia
Table saved ((1, 6)) at '/home/flavio/uv/perturb_agent/data/tcga/cases_for_PS_TCGA-BLCA_Subtype_acinar adenocarcinoma_Stage_stage iia.tsv'
1


,project_id,subtype,stage,case_id,n,frac
0,TCGA-BLCA,acinar adenocarcinoma,stage iia,97d6568c-0a89-4327-8988-71e81ee8dae6,1,1.0


In [16]:
force=False
verbose=True

print(pid, subtype, stage)

df_sample = gdc.get_samples(pid=pid, subtype=subtype, stage=stage, force=force, verbose=verbose)
print(len(df_sample))
df_sample

TCGA-BLCA acinar adenocarcinoma stage iia
Table saved ((3, 8)) at '/home/flavio/uv/perturb_agent/data/tcga/samples_for_PS_TCGA-BLCA_Subtype_acinar adenocarcinoma_Stage_stage iia.tsv'
Found 3 samples in many cases.
3


,project_id,subtype,stage,case_id,case_submitter_id,sample_id,sample_submitter_id,sample_type
0,TCGA-BLCA,acinar adenocarcinoma,stage iia,97d6568c-0a89-4327-8988-71e81ee8dae6,TCGA-YC-A8S6,edea764d-5123-4d05-be57-17a7c6f66f0d,TCGA-YC-A8S6-01Z,Primary Tumor
1,TCGA-BLCA,acinar adenocarcinoma,stage iia,97d6568c-0a89-4327-8988-71e81ee8dae6,TCGA-YC-A8S6,38a7d76d-1ec7-4469-a204-02f3ca423488,TCGA-YC-A8S6-01A,Primary Tumor
2,TCGA-BLCA,acinar adenocarcinoma,stage iia,97d6568c-0a89-4327-8988-71e81ee8dae6,TCGA-YC-A8S6,04fa063e-eeeb-4740-9412-504318ed173a,TCGA-YC-A8S6-10A,Blood Derived Normal


In [17]:
force=False
verbose=True

sample_ids = df_sample.sample_id.to_list()

print(pid, subtype, stage)

df_files = gdc.get_expression_files_given_samples(pid=pid, subtype=subtype, stage=stage, 
                                                  sample_ids=sample_ids, force=force, verbose=verbose)
print(len(df_files))
df_files

TCGA-BLCA acinar adenocarcinoma stage iia
Table saved ((1, 10)) at '/home/flavio/uv/perturb_agent/data/tcga/rnaseq_exp_files_for_PS_TCGA-BLCA_Subtype_acinar adenocarcinoma_Stage_stage iia.tsv'
Found 1 files.
1


,project_id,subtype,stage,case_id,file_id,file_name,case_submitter_id,sample_id,sample_submitter_id,workflow
0,TCGA-BLCA,acinar adenocarcinoma,stage iia,97d6568c-0a89-4327-8988-71e81ee8dae6,8e7ea117-cc02-4827-8be3-d99488389bb8,e76f88be-cbcf-499d-a9bb-e1c2c7ecef95.rna_seq.a...,TCGA-YC-A8S6,38a7d76d-1ec7-4469-a204-02f3ca423488,TCGA-YC-A8S6-01A,STAR - Counts


In [18]:
sample_ids

['edea764d-5123-4d05-be57-17a7c6f66f0d',
 '38a7d76d-1ec7-4469-a204-02f3ca423488',
 '04fa063e-eeeb-4740-9412-504318ed173a']